# FastRelax

## Goals

This tutorial runs a deliberately small FastRelax example and explains the protocol rather than presenting it as an exact Rosetta port. You will:

- build the required `PackerPalette`, move map, and `FoldForest`;
- run one repeat on a checked-in 1ubq slice;
- inspect the `fa_rep` and coordinate-constraint ramp;
- compare the score and structure before and after relaxation; and
- see the verified adapter signature for optional kinematic minimization.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from biotite.structure.io import load_structure

import tmol
from tmol import beta2016_score_function
from tmol.io.pose_stack_from_biotite import pose_stack_from_biotite
from tmol.kinematics.fold_forest import FoldForest
from tmol.kinematics.move_map import CartesianMoveMap, MoveMap
from tmol.optimization.minimizers import run_kin_min
from tmol.pack.packer_task import PackerPalette
from tmol.relax.fast_relax import fast_relax
from tmol.score.constraint.utility import constrain_all_ca
from tmol.score.score_types import ScoreType

SEED = 20260807
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def show_table(frame):
    try:
        from itables import show
    except ImportError:
        return display(frame)
    return show(frame)


def total_score(pose_stack, score_function):
    scorer = score_function.render_whole_pose_scoring_module(pose_stack)
    return float(scorer(pose_stack.coords).detach().cpu()[0])

## Protocol and ramp schedule

TMol's `fast_relax(pose_stack, sfxn, packer_pallete, move_map, fold_forest, *, ...)` alternates rotamer packing and minimization. Each schedule entry independently scales the starting `fa_rep` weight for packing and minimization and can scale the starting coordinate-constraint weight. The final minimization fraction should be `1.0` so the accepted structure is evaluated with full repulsion.

The production default is a four-step MonomerRelax2019-derived schedule. This tutorial uses two steps and one repeat to keep the checked-in example small. The default `min_fn=None` is Cartesian; therefore the workflow passes a `CartesianMoveMap`.

In [ ]:
repo_root = Path(tmol.__file__).resolve().parents[1]
cif_path = repo_root / "tmol" / "tests" / "data" / "cif" / "1UBQ.cif"
atom_array = load_structure(str(cif_path), model=1, include_bonds=True)
protein_slice = atom_array[(atom_array.chain_id == "A") & (atom_array.res_id <= 6)]
pose = pose_stack_from_biotite(protein_slice, device, no_optH=True)
relax_start = constrain_all_ca(pose)

score_function = beta2016_score_function(device)
score_function.set_weight(ScoreType.constraint, 1.0)
fa_rep_start = float(score_function.get_weight(ScoreType.fa_ljrep))
constraint_start = float(score_function.get_weight(ScoreType.constraint))

palette = PackerPalette()
cartesian_move_map = CartesianMoveMap()
fold_forest = FoldForest.reasonable_fold_forest(relax_start)
tiny_schedule = [
    {"fa_rep_pack_frac": 0.10, "fa_rep_min_frac": 0.20, "cst_frac": 1.0},
    {"fa_rep_pack_frac": 1.00, "fa_rep_min_frac": 1.00, "cst_frac": 0.0},
]

schedule_table = pd.DataFrame(tiny_schedule)
schedule_table["fa_rep_pack_weight"] = (
    schedule_table["fa_rep_pack_frac"] * fa_rep_start
)
schedule_table["fa_rep_min_weight"] = (
    schedule_table["fa_rep_min_frac"] * fa_rep_start
)
schedule_table["constraint_weight"] = schedule_table["cst_frac"] * constraint_start
print("input:", cif_path.name)
show_table(schedule_table)

## Run one repeat

The call below uses the complete current signature: pose, score function, palette, move map, and fold forest are positional; protocol controls are keyword-only. Coordinate constraints are already attached to `relax_start`, and the score function has a non-zero constraint weight, so `cst_frac` values in the schedule are active.

In [ ]:
score_before = total_score(relax_start, score_function)
relaxed = fast_relax(
    relax_start,
    score_function,
    palette,
    cartesian_move_map,
    fold_forest,
    num_repeats=1,
    schedule=tiny_schedule,
    ramp_constraints=True,
    verbose=False,
)
score_after = total_score(relaxed, score_function)

show_table(
    pd.DataFrame(
        [
            {"structure": "before", "weighted_score": score_before},
            {"structure": "after", "weighted_score": score_after},
        ]
    )
)

In [ ]:
tmol.switchable_view(
    {"before": relax_start, "after": relaxed},
    notes={
        "before": f"weighted score: {score_before:.3f}",
        "after": f"weighted score: {score_after:.3f}",
    },
)

## Optional kinematic minimizer

`fast_relax` calls a custom minimizer as `min_fn(pose_stack, sfxn, *, fold_forest, move_map, verbose)`. The adapter below uses the verified `run_kin_min(pose_stack, sfxn, ff, mm, ...)` signature. To use it, supply a `MoveMap` (not a `CartesianMoveMap`) and pass `min_fn=kinematic_min_fn` to `fast_relax`.

In [ ]:
def kinematic_min_fn(
    pose_stack, score_function, *, fold_forest, move_map, verbose
):
    return run_kin_min(
        pose_stack,
        score_function,
        fold_forest,
        move_map,
        optimizer_kwargs={"max_iter": 10, "verbose": verbose},
        verbose=verbose,
    )

kinematic_move_map = MoveMap.from_pose_stack(relax_start)
kinematic_move_map.move_all_named_torsions = True
# Example extension (not run here):
# kin_relaxed = fast_relax(
#     relax_start, score_function, palette, kinematic_move_map, fold_forest,
#     num_repeats=1, schedule=tiny_schedule, min_fn=kinematic_min_fn,
# )

## Rosetta comparison

Both protocols alternate side-chain repacking with minimization while ramping steric repulsion, and both can ramp coordinate constraints. Rosetta FastRelax has a mature script language, MoveMap factories, symmetry and membrane integrations, and many protocol options. TMol implements a smaller tensor-native subset: a Python schedule, `PackerTask` operations, Cartesian minimization by default, an optional callable minimizer, and accept-to-best across repeats.

The algorithms and score functions are not numerically interchangeable. Treat TMol FastRelax as partial protocol parity for batched refinement, not as an exact port or a guarantee of reproducing Rosetta trajectories.

## Limitations

- The two-step schedule is pedagogical, not a recommended production schedule.
- Results depend on the score function, available conformer samplers, device, dtype, and optimizer.
- FastRelax does not perform docking, backbone assembly, or global sampling.
- TMol restores the starting constraint weight after the protocol; it does not expose a full per-stage trajectory object.
- A custom `min_fn` is an extension point, so its compatibility and convergence behavior are the caller's responsibility.

## Exercises

1. Replace the tiny schedule with `DEFAULT_RELAX_SCHEDULE` and compare runtime and score.
2. Keep constraints active in the final step and compare Cα displacement.
3. Pass `kinematic_min_fn` with the prepared `kinematic_move_map` and compare Cartesian and kinematic results.
4. Write a task operation that restricts packing to a selected block mask.
5. Batch two small poses and inspect accept-to-best behavior per pose.

## References

- [Rosetta Relax tutorial](https://docs.rosettacommons.org/demos/latest/tutorials/Relax_Tutorial/Relax)
- [Rosetta FastRelax implementation](https://github.com/RosettaCommons/rosetta/blob/main/source/src/protocols/relax/FastRelax.cc)
- [PyRosetta packing, design, and regional relax](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/06.02-Packing-design-and-regional-relax.ipynb)
- [PyRosetta refinement documentation](https://www.pyrosetta.org/documentation)